# FIM-GAT v2: one job on Kaggle

This notebook runs one job of `kaggle/run_job.py`; `kaggle/README.md` in the bundle has the full guide.

Before **Save Version > Save & Run All**:
1. **Add Input**: the bundle dataset, and the outputs (`outputs_<job>.zip`) of the jobs this one needs.
2. **Settings**: Accelerator *GPU T4 x2* for the G jobs, *None* for the C jobs; Internet *on*.
3. Set `JOB` below.

When the version finishes, its **Output** holds `outputs_<JOB>.zip`. If the log ends with NOT FINISHED, run the same job again with that output attached: it resumes.

In [ ]:
# Which job to run (the order is in kaggle/README.md):
#   G1b-train             GPU T4 x2  constructor (attempt 2), start step, proposer  needs: nothing
#   C1-pokec-full-region  CPU        T1 on full Pokec, region                          needs: nothing
#   C1-pokec-full-gender  CPU        T1 on full Pokec, gender                          needs: nothing
#   C1-pokec-full-age     CPU        T1 on full Pokec, age                             needs: nothing
#   G2-learned            GPU T4 x2  the learned start's T1 rows, case studies         needs: G1b, the three C1
#   G3-t3                 GPU        T3 (timing)                                       needs: G1b, the three C1
#   G4-t4                 GPU        T4 (timing)                                       needs: G1b
JOB = "G1b-train"
HOURS = 11.0          # start no new work after this many hours; a Kaggle session ends at 12 h
SMOKE_TESTS = True    # run the fast test file first (about a minute)
SERIAL = False        # True: run the two GPU lanes one after another (if a session ran out of memory)

In [ ]:
# Setup: a writable copy of the bundle, checked against its manifest, and the package versions the cloud used.
import glob, hashlib, json, os, shutil, subprocess, sys, zipfile

WORK = "/tmp/fimgat"         # /kaggle/input is read-only; /kaggle/working is kept for the outputs
PINNED = ["torch_geometric==2.7.0", "numba==0.63.1", "numpy==2.3.5", "scipy==1.17.0", "scikit-learn==1.8.0",
          "pandas==3.0.0", "networkx==3.6.1", "psutil==7.2.1", "matplotlib==3.10.8", "pytest==9.0.3"]

def find_bundle():
    for m in sorted(glob.glob("/kaggle/input/**/BUNDLE_MANIFEST.json", recursive=True)):
        return "folder", os.path.dirname(m)          # Kaggle unpacked the uploaded zip
    for z in sorted(glob.glob("/kaggle/input/**/*.zip", recursive=True)):
        with zipfile.ZipFile(z) as zf:
            if "fimgat/BUNDLE_MANIFEST.json" in zf.namelist():
                return "zip", z
    raise SystemExit("No FIM-GAT bundle under /kaggle/input: Add Input > Your Datasets > the bundle dataset.")

kind, src = find_bundle()
shutil.rmtree(WORK, ignore_errors=True)
if kind == "folder":
    shutil.copytree(src, WORK)
else:
    with zipfile.ZipFile(src) as zf:
        zf.extractall("/tmp")                         # the zip holds one folder, fimgat/
subprocess.run(["chmod", "-R", "u+w", WORK], check=True)
with open(os.path.join(WORK, "BUNDLE_MANIFEST.json")) as f:
    bundle = json.load(f)
for rel, meta in bundle["extras"].items():
    with open(os.path.join(WORK, rel), "rb") as f:
        h = hashlib.sha1(f.read()).hexdigest()
    assert h == meta["sha1"], f"{rel} was damaged in the upload (sha1 {h}, expected {meta['sha1']})"
print(f"bundle: commit {bundle['commit'][:12]} of {bundle['created']} ({kind}: {src}); extra files verified")

pip = [sys.executable, "-m", "pip", "install", "-q", "--no-warn-conflicts"]
if subprocess.run(pip + PINNED).returncode != 0:
    print("WARNING: the pinned versions did not install (is Internet on? Settings > Internet).\n"
          "Falling back to the latest torch_geometric and numba with Kaggle's other packages; the run records the versions.")
    subprocess.run(pip + ["torch_geometric", "numba"])
check = subprocess.run([sys.executable, "-c", "import torch, torch_geometric, numba, numpy, pandas, scipy; "
                        "print('torch', torch.__version__, '| CUDA GPUs:', torch.cuda.device_count(), "
                        "'| torch_geometric', torch_geometric.__version__, '| numba', numba.__version__, "
                        "'| numpy', numpy.__version__, '| pandas', pandas.__version__, '| scipy', scipy.__version__)"],
                       capture_output=True, text=True)
print(check.stdout or check.stderr)
assert check.returncode == 0, "the environment cannot import the packages the code needs (see above)"

In [ ]:
# Run the job. Its own log lines appear here; each command's full log goes into the output zip.
cmd = [sys.executable, "-u", "kaggle/run_job.py", "--job", JOB, "--hours", str(HOURS),
       "--resume-from", "/kaggle/input", "--out", "/kaggle/working"] + (["--smoke-tests"] if SMOKE_TESTS else []) + (["--serial"] if SERIAL else [])
p = subprocess.Popen(cmd, cwd=WORK, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    print(line, end="", flush=True)
rc = p.wait()
print(f"\nrun_job.py exit code {rc} (0: finished; 1: not finished, run again with this output attached)")

In [ ]:
# What this version saves as its Output.
for f in sorted(os.listdir("/kaggle/working")):
    print(f"{os.path.getsize(os.path.join('/kaggle/working', f)) / 2**20:8.1f} MB  {f}")